In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [10]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [11]:
#Q1. How many lesson pages
len(documents)

72

In [12]:
#Q2. Indexing and searching
from minsearch import Index
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    # Use the actual fields exposed by RawRepositoryFile
    filename = getattr(file, "filename", None) or getattr(file, "path", "")
    content = getattr(file, "content", None)

    # If needed, parse explicitly
    if content is None:
        parsed = file.parse()
        content = getattr(parsed, "content", str(parsed))

    documents.append({
        "filename": filename,
        "content": content,
    })

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

if results:
    first = results[0]
    print("First result filename:", first["filename"])
    print("First result snippet:")
    print(first["content"][:300])
else:
    print("No results found.")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md
First result snippet:
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That wor


In [13]:
from dataclasses import dataclass
from typing import Any

from minsearch import Index
from openai import OpenAI


@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    search_results: list[dict[str, Any]]


class RAGBase:
    def __init__(
        self,
        index: Index,
        llm_client: OpenAI,
        model: str = "gpt-5.4-mini",
        instructions: str = """
You are a course teaching assistant.
Answer the question using only the provided context.
If the answer is not in the context, say "I don't know."
""",
        prompt_template: str = """
QUESTION: {question}

CONTEXT:
{context}
""".strip(),
    ):
        self.index = index
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.prompt_template = prompt_template

    def search(self, query: str, num_results: int = 5):
        # For our docs, we search over content and keep filename as metadata
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict={"content": 1.0},
        )

    def build_context(self, search_results: list[dict[str, Any]]) -> str:
        blocks = []
        for doc in search_results:
            blocks.append(f"Filename: {doc['filename']}")
            blocks.append(doc["content"])
            blocks.append("")
        return "\n".join(blocks).strip()

    def build_prompt(self, query: str, search_results: list[dict[str, Any]]) -> str:
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def _build_messages(self, prompt: str):
        return [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt},
        ]

    @staticmethod
    def _extract_input_tokens(response: Any) -> int:
        usage = getattr(response, "usage", None)
        if usage is None:
            return 0

        # OpenAI responses API often exposes input_tokens
        for attr in ("input_tokens", "prompt_tokens"):
            value = getattr(usage, attr, None)
            if value is not None:
                return int(value)

        # Some SDKs may expose usage as a dict
        if isinstance(usage, dict):
            for key in ("input_tokens", "prompt_tokens"):
                value = usage.get(key)
                if value is not None:
                    return int(value)

        return 0

    def llm(self, prompt: str):
        # Return the full response object so usage can be inspected
        return self.llm_client.responses.create(
            model=self.model,
            input=self._build_messages(prompt),
        )

    def rag(self, query: str) -> RAGResult:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        return RAGResult(
            answer=response.output_text,
            input_tokens=self._extract_input_tokens(response),
            search_results=search_results,
        )

In [14]:
import os
os.environ["OPENAI_API_KEY"]="<YOUR_OPENAI_API_KEY>"

from openai import OpenAI
openai_client = OpenAI()

In [15]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

# 1) Load lesson markdown docs
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    filename = getattr(file, "filename", None) or getattr(file, "path", "")
    content = getattr(file, "content", None)

    if content is None:
        parsed = file.parse()
        content = getattr(parsed, "content", str(parsed))

    documents.append({
        "filename": filename,
        "content": content,
    })

# 2) Build index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

# 3) Run RAG
client = OpenAI()
assistant = RAGBase(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
result = assistant.rag(query)

print("Answer:")
print(result.answer)

print("\nInput tokens sent to model:", result.input_tokens)

print("\nTop retrieved files:")
for doc in result.search_results[:5]:
    print("-", doc["filename"])

Answer:
The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items. If there were, it runs the tool, appends the tool result to the message history, and loops again.

It stops when the model returns a response with no function calls. The code uses a `has_function_calls` flag and breaks when that flag is `False`.

Input tokens sent to model: 7110

Top retrieved files:
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/15-frameworks.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/11-agents-intro.md
- 01-agentic-rag/lessons/16-other-frameworks.md


In [16]:
#Q4.Chunking
from gitsource import GithubRepositoryDataReader, chunk_documents

# 1) Read the lesson markdown files
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# 2) Convert them into documents with filename + content
documents = []
for file in files:
    filename = getattr(file, "filename", None) or getattr(file, "path", "")
    content = getattr(file, "content", None)

    if content is None:
        parsed = file.parse()
        content = getattr(parsed, "content", str(parsed))

    documents.append({
        "filename": filename,
        "content": content,
    })

# 3) Chunk the documents
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

# 4) Verify the result
print("Number of documents:", len(documents))
print("Number of chunks:", len(chunks))

# Optional: inspect the first few chunks
for i, chunk in enumerate(chunks[:8]):
    print(f"Chunk {i}: filename={chunk['filename']}, start={chunk.get('start')}, len={len(chunk['content'])}")

Number of documents: 72
Number of chunks: 295
Chunk 0: filename=01-agentic-rag/lessons/01-intro.md, start=0, len=2000
Chunk 1: filename=01-agentic-rag/lessons/01-intro.md, start=1000, len=2000
Chunk 2: filename=01-agentic-rag/lessons/01-intro.md, start=2000, len=1183
Chunk 3: filename=01-agentic-rag/lessons/02-environment.md, start=0, len=2000
Chunk 4: filename=01-agentic-rag/lessons/02-environment.md, start=1000, len=2000
Chunk 5: filename=01-agentic-rag/lessons/02-environment.md, start=2000, len=1680
Chunk 6: filename=01-agentic-rag/lessons/03-rag.md, start=0, len=2000
Chunk 7: filename=01-agentic-rag/lessons/03-rag.md, start=1000, len=2000


In [17]:
#Q5

import os
from pathlib import Path
from dataclasses import dataclass
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents

# Load API key from .env (if present)
load_dotenv(dotenv_path=Path(".env"))

# ------------------------------------------------------------
# Helper classes
# ------------------------------------------------------------
INSTRUCTIONS = """
You are a course teaching assistant.
Answer the question using only the provided context.
If the answer is not in the context, say "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    search_results: list[dict[str, Any]]


class RAGBase:
    def __init__(
        self,
        index: Index,
        llm_client: OpenAI,
        model: str = "gpt-5.4-mini",
    ):
        self.index = index
        self.llm_client = llm_client
        self.model = model

    def search(self, query: str, num_results: int = 5):
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict={"content": 1.0},
        )

    def build_context(self, search_results: list[dict[str, Any]]) -> str:
        blocks = []
        for doc in search_results:
            blocks.append(f"Filename: {doc['filename']}")
            if "start" in doc:
                blocks.append(f"Start: {doc['start']}")
            blocks.append(doc["content"])
            blocks.append("")
        return "\n".join(blocks).strip()

    def build_prompt(self, query: str, search_results: list[dict[str, Any]]) -> str:
        context = self.build_context(search_results)
        return PROMPT_TEMPLATE.format(question=query, context=context)

    @staticmethod
    def _extract_input_tokens(response: Any) -> int:
        usage = getattr(response, "usage", None)

        if usage is None:
            return 0

        # OpenAI responses API
        for key in ("input_tokens", "prompt_tokens"):
            value = getattr(usage, key, None)
            if value is not None:
                return int(value)

        # Some SDK versions expose usage as a dict
        if isinstance(usage, dict):
            for key in ("input_tokens", "prompt_tokens"):
                value = usage.get(key)
                if value is not None:
                    return int(value)

        return 0

    def llm(self, prompt: str):
        return self.llm_client.responses.create(
            model=self.model,
            input=[
                {"role": "developer", "content": INSTRUCTIONS},
                {"role": "user", "content": prompt},
            ],
        )

    def rag(self, query: str) -> RAGResult:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)

        return RAGResult(
            answer=response.output_text,
            input_tokens=self._extract_input_tokens(response),
            search_results=search_results,
        )


# ------------------------------------------------------------
# Load lesson documents
# ------------------------------------------------------------
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    filename = getattr(file, "filename", None) or getattr(file, "path", "")
    content = getattr(file, "content", None)

    if content is None:
        parsed = file.parse()
        content = getattr(parsed, "content", str(parsed))

    documents.append({
        "filename": filename,
        "content": content,
    })

# ------------------------------------------------------------
# Chunked index
# ------------------------------------------------------------
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# ------------------------------------------------------------
# Query
# ------------------------------------------------------------
query = "How does the agentic loop keep calling the model until it stops?"

client = OpenAI()
assistant = RAGBase(index=chunk_index, llm_client=client)

result = assistant.rag(query)

print("Query:", query)
print("Chunked input tokens:", result.input_tokens)
print("Top retrieved chunks:")
for doc in result.search_results[:5]:
    print("-", doc["filename"], "| start =", doc.get("start"))

print("\nAnswer preview:")
print(result.answer[:800])

Query: How does the agentic loop keep calling the model until it stops?
Chunked input tokens: 2322
Top retrieved chunks:
- 01-agentic-rag/lessons/14-agentic-loop.md | start = 4000
- 01-agentic-rag/lessons/14-agentic-loop.md | start = 5000
- 01-agentic-rag/lessons/14-agentic-loop.md | start = 0
- 01-agentic-rag/lessons/15-frameworks.md | start = 4000
- 01-agentic-rag/lessons/15-frameworks.md | start = 3000

Answer preview:
It keeps calling the model inside a `while True` loop. Each turn:

- it sends the current `messages` to the model,
- checks the model output for any `function_call`,
- if there is one, it runs the tool, appends the tool result, and continues,
- if there are no function calls, it breaks.

The loop stops when `has_function_calls == False`, meaning the model returned a final answer with no more tool calls.


In [ ]:
#Q6
from pathlib import Path
from typing import Any
from dataclasses import dataclass

from dotenv import load_dotenv
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Load API key if you keep it in a .env file
load_dotenv(dotenv_path=Path(".env"), override=True)

# ------------------------------------------------------------
# Load markdown lesson docs and build chunked index
# ------------------------------------------------------------
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    filename = getattr(file, "filename", None) or getattr(file, "path", "")
    content = getattr(file, "content", None)

    if content is None:
        parsed = file.parse()
        content = getattr(parsed, "content", str(parsed))

    documents.append({
        "filename": filename,
        "content": content,
    })

chunks = chunk_documents(documents, size=2000, step=1000)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# ------------------------------------------------------------
# Tool wrapper with call counter
# ------------------------------------------------------------
search_call_count = 0

def search(query: str) -> list[dict[str, Any]]:
    """
    Search the chunked lesson documents for passages relevant to the query.
    """
    global search_call_count
    search_call_count += 1

    return chunk_index.search(
        query,
        num_results=5,
        boost_dict={"content": 1.0},
    )

# ------------------------------------------------------------
# Agent setup
# ------------------------------------------------------------
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

runner = OpenAIResponsesRunner(
    instructions=instructions,
    model="gpt-5.4-mini",
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
    callback=DisplayingRunnerCallback(),
)

runner.add_function(search)

question = (
    "How does the agentic loop work, and how is it different from plain RAG?"
)

response = runner.run(question)

print("\n=== Result ===")
print(response)
print(f"\nSearch tool was called {search_call_count} times.")

In [22]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback, OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)   # or agent_tools.add_tools(...)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

print(result.cost)
print(result.all_messages)

-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.0084375'), output_cost=Decimal('0.001575'), total_cost=Decimal('0.0100125'))
[EasyInputMessage(content="\nYou're a course teaching assistant.\nAnswer the student's question using the search tool.\nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Olama local run install Ollama locally"}', call_id='call_etm38QLot9mcuV6Rx7BjFJPd', name='search', type='function_call', id='fc_0411399e1fcaceca006a370743e59881a292e40b6aeddb3c6c', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"run Ollama locally command ollama serve pull model"}', call_id='call_KTNZA7YKakzhYEzrbof2SMLG', name='search', type='function_call', id='fc_0411399e1fcaceca006a370743e5a881a2abb14917a879b14c', namespace=None, status='completed'), ResponseFunctionToolCall(arg

In [ ]:
# 4 times